# Asking for Properties: the Data and Math Behind a Conditional Transformer VAE
**Series B: Deep Learning for Molecules, Post 6b** &nbsp;|&nbsp; companion *primer* notebook

This is the lightweight companion to the full tutorial notebook `TCVAE_Tutorial.ipynb` (GPU runtime, 92 MB pre-trained model).
**This primer needs neither.** It runs on a CPU in well under a minute and lets you check the data and the mathematics behind the post.

| Section | What you verify |
|---|---|
| 1 | turning LogP, QED and SAS into 50 classes and 150 condition tokens; how sparse the classes are |
| 2 | where the 23,046,693 parameters live, and what the condition adds to the unconditional model |
| 3 | what the model's simplified MMD penalty can and cannot see (only the *length* of $z$) |
| 4 | a back-of-envelope model of why the latent vector may compete with the condition |
| 5 | how well the requested LogP is honoured: bias, shrinkage, spread, hit rate |
| 6 | the sampling metrics of the conditional model, with error bars |

**How to use it:** run top to bottom (**Runtime → Run all**), try the **Stop and predict** cells, then do the exercises.
Numbers about the *trained model* (scan results, validity, probe $R^2$) are typed in from the saved run of the full tutorial; here we analyse them.

## 0. Setup and data

In [ ]:
import os, math, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

CSV_NAME = "250k_rndm_zinc_drugs_clean_3.csv"
CSV_URL = ("https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/"
           "models/zinc_properties/250k_rndm_zinc_drugs_clean_3.csv")

if not os.path.exists(CSV_NAME):
    try:
        print("Downloading the ZINC data set (about 22 MB) ...")
        urllib.request.urlretrieve(CSV_URL, CSV_NAME)
    except Exception as err:
        print("Download failed:", err)
        try:
            from google.colab import files
            print("Please select", CSV_NAME)
            up = files.upload(); os.replace(next(iter(up)), CSV_NAME)
        except ImportError:
            raise FileNotFoundError(f"Place {CSV_NAME} next to this notebook and re-run.")

csv = pd.read_csv(CSV_NAME)
props = csv[["logP", "qed", "SAS"]].values            # LogP, QED, SAS of every molecule
print(csv.shape, list(csv.columns))

## 1. From properties to condition tokens
Each property is cut into **50 equal-width classes** between its smallest and largest value; class $k$ of property $p$ becomes the token $\text{base}_p+k$ with bases 40 (LogP), 90 (QED) and 140 (SAS).
**Stop and predict:** the post says the first molecule gets tokens `[79, 125, 147]`. Which classes is that?

In [ ]:
NAMES = ("LogP", "QED", "SAS"); BASE = (40, 90, 140)

def class_edges(v):                                   # lower edge of each of the 50 classes
    lo, hi = v.min(), v.max()
    return np.linspace(lo, hi - (hi - lo) / 50, 50)

edges = [class_edges(props[:, p]) for p in range(3)]
width = [e[1] - e[0] for e in edges]

cats = np.zeros((len(props), 3), dtype=np.int64)      # condition tokens of every molecule
for p in range(3):
    n_below = (props[:, p][:, None] > edges[p][None, :]).sum(1)          # class edges below the value
    cats[:, p] = BASE[p] + np.clip(n_below, 1, 50) - 1
cls = cats - np.array(BASE)                           # class numbers 0..49

for p in range(3):
    print(f"{NAMES[p]:5s} range [{props[:, p].min():.3f}, {props[:, p].max():.3f}]   class width {width[p]:.4f}")
print("tokens of the first three molecules:", cats[:3].tolist())

def class_range(p, k): return edges[p][k], edges[p][k] + width[p]
def describe(c):
    return ", ".join(f"{NAMES[p]} in [{class_range(p, k)[0]:.2f}, {class_range(p, k)[1]:.2f}]" for p, k in enumerate(c))
print("\nclasses 39 / 35 / 7 ->", describe((39, 35, 7)))
print("LogP class 10, QED and SAS class 10 ->", describe((10, 10, 10)))
print("LogP class 40 ->", class_range(0, 40))

### How sparse are the classes?
Equal-width classes are simple, but the property distributions are far from flat.

In [ ]:
ks = np.arange(5, 50, 5)
counts = np.array([(cls[:, 0] == k).sum() for k in ks])
print("LogP class        :", ks.tolist())
print("molecules in class:", counts.tolist())

keys = cls[:, 0] * 10000 + cls[:, 1] * 100 + cls[:, 2]
_, cell_counts = np.unique(keys, return_counts=True)
print(f"\njoint cells (LogP, QED, SAS classes): {50**3:,} possible, {len(cell_counts):,} occupied, "
      f"median {int(np.median(cell_counts))} molecules per occupied cell")
for trip in ((39, 35, 7), (10, 10, 10), (10, 21, 31), (40, 10, 10), (40, 22, 11)):
    print(f"  exact request {trip}: {int((cls == np.array(trip)).all(1).sum())} molecules")

print("\ncorrelations between the properties:")
print(pd.DataFrame(np.corrcoef(props.T), index=NAMES, columns=NAMES).round(3))

plt.figure(figsize=(7.5, 3.4))
plt.bar(range(50), np.bincount(cls[:, 0], minlength=50), color=COLORS["gray"]); plt.yscale("log")
for k in ks: plt.axvline(k, color=COLORS["orange"], lw=0.8)
plt.xlabel("LogP class"); plt.ylabel("molecules (log scale)"); plt.title("Most classes at the edges are nearly empty"); plt.show()

### A realistic request
The properties are correlated, so asking for LogP class 10 with QED and SAS at *arbitrary* classes may ask for a combination that hardly exists.
`typical_condition` instead takes the median QED and SAS classes among the molecules of the requested LogP class (or the nearest populated class).

In [ ]:
def typical_condition(prop, k):
    populated = np.unique(cls[:, prop])
    k_used = k if k in populated else int(populated[np.argmin(np.abs(populated - k))])
    rows = cls[cls[:, prop] == k_used]
    cond = [int(np.median(rows[:, p])) for p in range(3)]
    cond[prop] = k
    return tuple(cond)

typical = {k: typical_condition(0, k) for k in ks}
print(typical)
print("LogP class 10 ->", describe(typical[10]))
print("LogP class 40 ->", describe(typical[40]))

## 2. Where do the 23,046,693 parameters live?
Derive the total from the architecture: 190 embedding entries (tokens 0 to 189), 200 position slots, embedding size 128, 6 + 6 layers, an encoder row of $115\times128$ numbers squeezed to 1280, and a decoder input of $1280+3\times128=1664$ numbers.

In [ ]:
H, PF, LAYERS = 128, 256, 6
lin = lambda m, n: m * n + n
ln  = lambda n: 2 * n
attn = 4 * lin(H, H); ff = lin(H, PF) + lin(PF, H)
enc_layer = 2 * ln(H) + attn + ff
dec_layer = 3 * ln(H) + 2 * attn + ff

IN_VOCAB, OUT_VOCAB, POS = 190, 37, 200          # encoder knows 190 tokens; the decoder only ever writes the 37 SMILES tokens
ROW, LATENT, N_COND = 115, 1280, 3
parts = {
    "encoder embeddings (190 tokens + 200 positions)": IN_VOCAB * H + POS * H,
    "encoder layers (6)":                              LAYERS * enc_layer,
    "decoder embeddings (37 tokens + 200 positions)":  OUT_VOCAB * H + POS * H,
    "decoder layers (6)":                              LAYERS * dec_layer,
    "decoder output layer (128 -> 37)":                lin(H, OUT_VOCAB),
    "latent layer (14,720 -> 1,280)":                  lin(ROW * H, LATENT),
    "inverse layer (1,664 -> 1,280)":                  lin(LATENT + N_COND * H, LATENT),
}
total = sum(parts.values())
for k, v in parts.items():
    print(f"{k:50s} {v:>12,d}  {100 * v / total:5.1f}%")
print(f"{'TOTAL':50s} {total:>12,d}")
print("matches the number printed by the full tutorial (23,046,693):", total == 23_046_693)

# what did the condition add, relative to the unconditional model of the previous post (21,859,969)?
uncond = 21_859_969
diff = {"larger encoder embedding table (190 vs 37 tokens, 200 vs 120 positions)": (IN_VOCAB * H + POS * H) - (37 * H + 120 * H),
        "decoder position table (200 vs 120 positions)":                           (OUT_VOCAB * H + POS * H) - (37 * H + 120 * H),
        "wider latent layer (115 vs 111 positions)":                               lin(ROW * H, LATENT) - lin(111 * H, LATENT),
        "wider inverse layer (1,664 vs 1,280 inputs)":                             lin(LATENT + N_COND * H, LATENT) - lin(LATENT, LATENT),
        "residual blocks (removed)":                                               -2 * lin(10, 10)}
print()
for k, v in diff.items(): print(f"{v:>+12,d}  {k}")
print(f"{sum(diff.values()):>+12,d}  total change  ->  {total - uncond:+,d} = {total:,} - {uncond:,}:", sum(diff.values()) == total - uncond)

## 3. What can the simplified penalty see?
As in the previous post, the loss contains $w\,[1-\overline{k(z,y)}]$ with $k(x,y)=\exp(-\lVert x-y\rVert^2/s)$, $s=d^2\cdot0.0005=819.2$, $d=1280$, and prior samples $y\sim N(0,I)$. This model uses weight $w=4$ (the unconditional one used 8).

Because $y$ is an isotropic Gaussian, the average of $k(z,y)$ over $y$ depends on the vector $z$ **only through its length**:
$$E_y\,k(z,y)=\Big(1+\tfrac2s\Big)^{-d/2}\exp\!\Big(-\frac{\lVert z\rVert^2}{s+2}\Big).$$
**Stop and predict:** the full tutorial measured $\overline{\lVert z\rVert}=11.8$ for encoded molecules, against 35.8 for the prior. What penalty value does the formula give? (Measured: 3.282, with 3.820 for two prior samples.)

In [ ]:
d = 1280; s = d * d * 0.0005; w = 4

def penalty_from_length(L, w=w):                    # penalty for a latent vector of length L
    return w * (1 - (1 + 2 / s) ** (-d / 2) * math.exp(-L**2 / (s + 2)))

prior_vs_prior = w * (1 - (1 + 4 / s) ** (-d / 2))  # both samples from N(0, I): the lengths are ~ sqrt(d)
print(f"prior vs prior : {prior_vs_prior:.3f}   (measured 3.820)")
print(f"|z| = 11.8     : {penalty_from_length(11.8):.3f}   (measured 3.282)")
print(f"|z| = 0        : {penalty_from_length(0.0):.3f}   <- the minimum")

lo, hi = 0.0, 40.0                                   # which length reproduces the measured value exactly?
for _ in range(60):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if penalty_from_length(mid) < 3.282 else (lo, mid)
print(f"the measured 3.282 corresponds to |z| = {lo:.1f}   (the tutorial measured a mean of 11.8)")
print(f"\nprevious post, w = 8, |z| = 14.3: {penalty_from_length(14.3, 8):.3f}   (measured 6.672)")

Now check the claim that the penalty is **blind to direction**. We build two sets of latent vectors with the *same* length (11.8) but very different shapes: one isotropic, one squeezed into a few directions (like the real encoder output, whose first principal component has a large variance). Their penalties against the prior should agree.

In [ ]:
rng = np.random.default_rng(0)
n = 500
Y = rng.standard_normal((1000, d))                   # prior samples

def kern(X, Y):
    D = (X**2).sum(1)[:, None] + (Y**2).sum(1)[None, :] - 2 * X @ Y.T
    return np.exp(-np.maximum(D, 0) / s)

def penalty_mc(Z): return w * (1 - kern(Z, Y).mean())

Z_iso = rng.standard_normal((n, d))
Z_iso *= 11.8 / np.linalg.norm(Z_iso, axis=1, keepdims=True)                 # isotropic direction, length 11.8

scales = np.exp(2.0 * rng.standard_normal(d))                                # very uneven coordinate scales
Z_ani = rng.standard_normal((n, d)) * scales
Z_ani *= 11.8 / np.linalg.norm(Z_ani, axis=1, keepdims=True)                 # same length, squeezed shape
share = (Z_ani**2).mean(0); share = np.sort(share)[::-1]
print(f"anisotropic set: the 20 largest of 1280 coordinates carry {100 * share[:20].sum() / share.sum():.0f}% of the variance")

print(f"penalty, isotropic set     : {penalty_mc(Z_iso):.3f}")
print(f"penalty, anisotropic set   : {penalty_mc(Z_ani):.3f}")
print(f"formula, length 11.8       : {penalty_from_length(11.8):.3f}")

**Why the principal-component test can mislead.** A first-component histogram of encoded vectors can look just like the prior's, while the vectors are three times too short. In the full tutorial, Figure 6a looks fine and Figure 6c does not. The null reference for the first component with $n$ points is $1+\sqrt{d/n}$:

In [ ]:
for n_ in (256, 2000):
    A = rng.standard_normal((n_, d)); A -= A.mean(0)
    print(f"n = {n_:>4}: first-PC std of perfect N(0,I) data = {np.linalg.svd(A, compute_uv=False)[0] / math.sqrt(n_ - 1):.2f}  "
          f"(prediction 1 + sqrt(d/n) = {1 + math.sqrt(d / n_):.2f})")

# length of the decoder's training input z + noise, versus the prior
sz = 0.33
print(f"\nencoder output length ~ {math.sqrt(d) * sz:.1f} (measured 11.8) | decoder training input ~ {math.sqrt(d * (sz**2 + 1)):.1f} "
      f"| prior draw ~ {math.sqrt(d):.1f}")

## 4. Does the latent vector compete with the condition?
The full tutorial fits a ridge regression that predicts each property from $z$ of *real* molecules: held-out $R^2$ is **0.617** (LogP), **0.624** (QED) and **0.742** (SAS), against $\approx0$ for shuffled labels. So $z$ knows the properties, and the decoder could in principle read them from $z$ instead of from the condition $c$.

During training, however, the decoder sees $z+\varepsilon$ with unit noise, while $c$ arrives clean. A back-of-envelope model: suppose the property enters $z$ as a signal along one direction, on top of other variation of variance $v=\sigma_z^2$ per coordinate. The best linear readout then has $R^2=\mathrm{SNR}/(1+\mathrm{SNR})$, and adding unit noise multiplies the SNR by $v/(v+1)$.
**Stop and predict:** by how much does the noise cut the readable property information?

In [ ]:
v = 0.33**2                                          # per-coordinate variance of the encoder output (measured std 0.33)
def r2_with_noise(r2_clean, v=v, noise=1.0):
    snr = r2_clean / (1 - r2_clean)
    snr_noisy = snr * v / (v + noise)
    return snr_noisy / (1 + snr_noisy)

print(f"{'property':8s} {'R2 from clean z':>16s} {'toy R2 from z + unit noise':>28s}")
for name, r2 in (("LogP", 0.617), ("QED", 0.624), ("SAS", 0.742)):
    print(f"{name:8s} {r2:>16.3f} {r2_with_noise(r2):>28.3f}")

# how much noise would be needed to push the LogP probe down to R2 = 0.05?
for noise in (0.0, 0.1, 0.5, 1.0, 2.0, 5.0):
    print(f"noise variance {noise:>4}: toy R2 for LogP = {r2_with_noise(0.617, noise=noise):.3f}")

This is a **hypothesis generator, not a result**: it assumes an isotropic residual and a linear readout. Its testable prediction (Exercise 5 of the post): when you *edit* a real molecule (keep its $z$, change the LogP class), adding noise to $z$ before decoding should make the decoder follow the request more strongly.

## 5. How well is the requested LogP honoured?
The table below is typed in from the systematic scan of the full tutorial: for each requested LogP class, 300 molecules were generated at random latent vectors and their LogP was measured with RDKit. **Typical** means that QED and SAS were set to the typical classes of Section 1; **fixed** means both were set to class 10.

In [ ]:
scan = pd.DataFrame({
    "class":       ks,
    "requested":   [-5.21, -3.70, -2.19, -0.67, 0.84, 2.35, 3.86, 5.38, 6.89],     # centre of the requested class
    "n_in_class":  counts,                                                           # from Section 1 (all 249,455 molecules)
    "fixed_mean":  [-1.55, -1.94, -1.17, -0.09, 1.47, 2.63, 3.82, 5.14, 5.75],
    "fixed_std":   [1.06, 1.01, 1.05, 0.93, 0.93, 0.85, 0.77, 0.74, 0.76],
    "fixed_valid": [83.3, 83.7, 87.7, 87.0, 90.0, 86.7, 84.0, 78.7, 79.3],
    "typ_mean":    [-2.27, -2.53, -1.75, -0.44, 1.03, 2.32, 3.71, 4.96, 5.63],
    "typ_std":     [1.04, 1.00, 0.89, 0.84, 0.80, 0.71, 0.67, 0.71, 0.75],
    "typ_valid":   [92.0, 91.7, 93.0, 94.0, 95.7, 93.3, 94.3, 83.3, 81.7]})
scan["fixed_bias"] = scan.fixed_mean - scan.requested
scan["typ_bias"]   = scan.typ_mean - scan.requested
print(scan[["class", "requested", "n_in_class", "typ_mean", "typ_bias", "typ_std", "typ_valid"]].round(2).to_string(index=False))

dense = scan["class"].between(20, 35)
for name in ("fixed", "typ"):
    b = scan[f"{name}_bias"]
    slope_all = np.polyfit(scan.requested, scan[f"{name}_mean"], 1)[0]
    slope_mid = np.polyfit(scan.requested[2:8], scan[f"{name}_mean"][2:8], 1)[0]
    print(f"{name:6s}: mean |bias| all classes {b.abs().mean():.2f}, classes 20-35 {b[dense].abs().mean():.2f} | "
          f"slope achieved-vs-requested: all {slope_all:.2f}, classes 15-40 {slope_mid:.2f}")
print(f"correlation of |bias| with log10(molecules in class), typical: "
      f"{np.corrcoef(scan.typ_bias.abs(), np.log10(scan.n_in_class))[0, 1]:.2f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot(scan.requested, scan.requested, "k--", lw=1, label="perfect control")
ax[0].errorbar(scan.requested, scan.fixed_mean, yerr=scan.fixed_std, fmt="o-", ms=4, capsize=3, color=COLORS["orange"], label="others fixed at class 10")
ax[0].errorbar(scan.requested, scan.typ_mean,   yerr=scan.typ_std,   fmt="o-", ms=4, capsize=3, color=COLORS["blue"],   label="others typical")
ax[0].set(xlabel="requested LogP (class centre)", ylabel="LogP of generated molecules (mean +/- std)", title="(a) Requested vs achieved"); ax[0].legend(fontsize=8)
ax[1].plot(scan["class"], scan.typ_bias, "o-", color=COLORS["blue"]); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(xlabel="requested LogP class", ylabel="bias = mean achieved - requested", title="(b) Bias: pulled toward the data mean")
ax[2].bar(scan["class"].astype(str), scan.n_in_class, color=COLORS["gray"]); ax[2].set_yscale("log")
ax[2].set(xlabel="requested LogP class", ylabel="molecules in class (log)", title="(c) Data per class")
plt.tight_layout(); plt.show()

### Spread, hit rate and the cost of an exact request
Even with no bias, the generated LogP values *spread* around the request. The class is only 0.30 LogP units wide. If the generated values are roughly normal with the measured mean and spread, the probability that a valid molecule falls **inside the requested class** is $\Phi\!\big(\tfrac{h-b}{\sigma}\big)-\Phi\!\big(\tfrac{-h-b}{\sigma}\big)$ with half-width $h=0.15$, bias $b$ and spread $\sigma$. (This uses the normal approximation, which we did not check.)
**Stop and predict:** what fraction of molecules land in the requested class in the best case?

In [ ]:
from math import erf, sqrt
Phi = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
Delta = width[0]; half = Delta / 2
print(f"class width {Delta:.3f}; a uniform spread inside one class would have std {Delta / sqrt(12):.3f}\n")
print(f"{'class':>5} {'bias':>6} {'std':>5} {'std/width':>9} {'P(in class)':>12} {'draws per hit':>14}")
for _, r in scan.iterrows():
    hit = Phi((half - r.typ_bias) / r.typ_std) - Phi((-half - r.typ_bias) / r.typ_std)
    draws = 1 / (hit * r.typ_valid / 100)
    print(f"{int(r['class']):>5} {r.typ_bias:>+6.2f} {r.typ_std:>5.2f} {r.typ_std / Delta:>9.1f} {hit:>12.3f} {draws:>14.1f}")

### Editing a molecule: the cleanest test, and a weak response
In the full tutorial's editing test (Figure 10), a real molecule keeps its latent vector and is decoded again with a LogP class shifted by $\pm5$ or $\pm10$ classes (that is, $\pm1.5$ or $\pm3.0$ LogP units). Reading the plotted means off the figure (approximate): a requested change of $-3.0$ gave about $-0.9$, $-1.5$ gave about $-0.3$, and $+1.5$ and $+3.0$ gave about $+0.3$ and $+0.8$.

In [ ]:
asked = np.array([-3.0, -1.5, 0.0, 1.5, 3.0])
got   = np.array([-0.93, -0.27, -0.03, 0.27, 0.83])           # read off Figure 10a of the full tutorial: approximate
slope_edit = np.polyfit(asked, got, 1)[0]
print(f"editing real molecules (same z): achieved change = {slope_edit:.2f} x requested change   (approximate)")
print(f"sampling at random z (Section 5): slope {np.polyfit(scan.requested[2:8], scan.typ_mean[2:8], 1)[0]:.2f} over classes 15-40")

## 6. The sampling metrics of the conditional model
5000 samples with the typical condition (classes 39 / 35 / 7), and 5000 test molecules reconstructed, from the saved run.

In [ ]:
def percent_with_ci(k, n):
    p = k / n
    return 100 * p, 100 * 1.96 * math.sqrt(p * (1 - p) / n)

rows = [("Parses as SMILES (grammar only)",        4975, 5000),
        ("Chemically valid",                        4882, 5000),
        ("Repeated (among valid)",                  21,   4882),
        ("Novel (among distinct valid)",            4857, 4861),
        ("Exactly reconstructed (test molecules)",  2660, 5000)]
table = pd.DataFrame([(m, k, n, *percent_with_ci(k, n)) for m, k, n in rows],
                     columns=["Metric", "count", "out of", "percent", "95% CI (+/- points)"])
print(table.round(2).to_string(index=False))

valid, distinct, novel = 4882, 4882 - 21, 4857
print(f"\nuniqueness {100 * distinct / valid:.2f}%  |  yield of new, distinct, valid molecules {novel}/5000 = {100 * novel / 5000:.2f}%")
print("unconditional model of the previous post: valid 79.42%, reconstructed 68.58%, yield 76.06%")
print(f"\nBut validity depends on the request: in the scan it ranges from {scan.typ_valid.min():.1f}% to {scan.typ_valid.max():.1f}% "
      f"(typical) and from {scan.fixed_valid.min():.1f}% to {scan.fixed_valid.max():.1f}% (fixed).")

## 7. Your turn
**Exercise 1 (tokens).** A molecule has LogP $=2.5$, QED $=0.75$, SAS $=2.2$. Compute its three classes and tokens by hand, then check with the code of Section 1.

**Exercise 2 (parameters).** If the condition used 100 classes per property, how many parameters would the embedding tables add? (Hint: the encoder table has one row per token.)

**Exercise 3 (the penalty).** Compute the penalty for $\lVert z\rVert=5,\ 20,\ 35.8$ with $w=4$. Which length does the penalty prefer?

**Exercise 4 (noise).** In the toy model of Section 4, how large must the noise variance be to push the QED probe $R^2$ below 0.05?

**Exercise 5 (hit rate).** Using Section 5, how many draws are needed on average to obtain 100 molecules inside LogP class 35? Inside class 10? Why is the class-10 number an optimistic estimate?

**Exercise 6 (sparsity).** The "typical" QED and SAS classes for LogP class 45 are medians over only 7 molecules. How reliable are they? Look at the spread of the QED and SAS classes among those 7 molecules, and compare with LogP class 30.

In [ ]:
# Starters
# Exercise 1
vals = (2.5, 0.75, 2.2)
for p in range(3):
    k = int(np.clip((edges[p] < vals[p]).sum(), 1, 50)) - 1
    print(f"{NAMES[p]}: class {k}, token {BASE[p] + k}")

# Exercise 2: embedding table sizes with 100 classes per property
print("extra encoder embedding rows for 300 vs 150 condition tokens:", (300 - 150) * H, "parameters")

# Exercise 3
print({L: round(penalty_from_length(L), 3) for L in (5, 20, 35.8)})

# Exercise 4
for noise in (1, 2, 5, 10, 20):
    print(f"noise {noise:>2}: toy QED probe R2 = {r2_with_noise(0.624, noise=noise):.3f}")

# Exercise 6: how much do QED and SAS classes vary among the molecules of a LogP class?
for k in (45, 30):
    rows = cls[cls[:, 0] == k]
    print(f"LogP class {k}: {len(rows):>5} molecules | QED class: median {np.median(rows[:, 1]):.0f}, std {rows[:, 1].std():.1f}"
          f" | SAS class: median {np.median(rows[:, 2]):.0f}, std {rows[:, 2].std():.1f}")

## 8. Self-check

In [ ]:
assert len(csv) == 249_455 and cats[:3].tolist() == [[79, 125, 147], [73, 138, 158], [79, 119, 150]]
assert counts.tolist() == [4, 37, 406, 2380, 9368, 20044, 16067, 1328, 7]
assert typical[10] == (10, 21, 31) and typical[40] == (40, 22, 11) and typical[45] == (45, 12, 12)
assert (len(cell_counts), int((cls == np.array([39, 35, 7])).all(1).sum())) == (22_148, 4)
assert total == 23_046_693 and total - uncond == 1_186_724
assert abs(prior_vs_prior - 3.820) < 0.01 and abs(penalty_from_length(11.8) - 3.282) < 0.02
assert abs(penalty_mc(Z_iso) - penalty_mc(Z_ani)) < 0.02 and abs(penalty_mc(Z_iso) - penalty_from_length(11.8)) < 0.02
assert abs(r2_with_noise(0.617) - 0.137) < 0.002
assert abs(scan.typ_bias.abs().mean() - 0.76) < 0.01 and abs(np.corrcoef(scan.typ_bias.abs(), np.log10(scan.n_in_class))[0, 1] + 0.875) < 0.01
assert round(percent_with_ci(4882, 5000)[1], 2) == 0.42 and round(percent_with_ci(2660, 5000)[0], 2) == 53.2
print("All checks passed: you reproduced the numbers in the post.")